# Unsloth GRPO Training for Hierarchical Reasoning

This notebook uses Unsloth's optimized kernels with TRL's GRPOTrainer for stable training.

**Key features:**
- ~50% less VRAM usage than standard transformers
- vLLM fast inference for generation
- HICRA-inspired reward functions for reasoning

In [4]:
# Cell 1: Environment Setup
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extra 30% context lengths
os.environ["fix_mistral_regex"] = "True"
# os.environ["OMP_NUM_THREADS"] = "1"

# Install dependencies (run this if not already installed)
# !pip install unsloth vllm
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [5]:
# Cell 2: HuggingFace Login
import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in with HF_TOKEN")
else:
    login()
    print("✅ Logged in interactively")

✅ Logged in with HF_TOKEN


In [6]:
from unsloth import FastLanguageModel
import torch
# Configuration
max_seq_length = 1024  # Can increase for longer reasoning traces
lora_rank = 32  # Larger rank = smarter, but slower
print("⏳ Loading model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,  # Disabled - requires CUDA toolkit for vLLM
)
print("🔗 Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Optimized gradient checkpointing
    random_state=3407,
)
print("✅ Model loaded successfully!")

⏳ Loading model with Unsloth...
==((====))==  Unsloth 2025.12.8: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.594 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔗 Attaching LoRA adapters...


Unsloth 2025.12.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model loaded successfully!


Tuning Tips:
- If you're getting OOM: Lower NEMOTRON_SAMPLE_SIZE to 1000-2000
- If generations are too long: Lower MAX_ANSWER_TOKENS to 400
- If you want more data: Increase NEMOTRON_SAMPLE_SIZE to 5000+
The filtering keeps ~60-70% of examples typically, so 3000 samples → ~2000 usable examples mixed with your 729 HICRA examples.



In [7]:
# Cell 4: Load and Combine Datasets
from datasets import load_dataset, Dataset
import json

# === Configuration ===
MAX_PROMPT_TOKENS = 400    # Filter out prompts longer than this
MAX_ANSWER_TOKENS = 600    # Filter out answers longer than this  
NEMOTRON_SAMPLE_SIZE = 3000  # How many Nemotron examples to use

# System prompt for reasoning format
SYSTEM_PROMPT = """
You are a mathematical reasoning assistant. Think through problems step by step.
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def format_prompt(example):
    """Format dataset for GRPO training with chat template."""
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
            {'role': 'user', 'content': example['prompt']}
        ],
        'answer': str(example['answer'])
    }

def format_nemotron(example):
    """Convert Nemotron format to our format."""
    messages = example.get('messages', [])
    
    # Extract user prompt and assistant answer
    user_content = ""
    assistant_content = ""
    
    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant':
            assistant_content = msg['content']
    
    # Get expected answer (fallback to assistant content if not available)
    expected = example.get('expected_answer', '')
    if not expected:
        # Try to extract from assistant's <answer> tags if present
        if '<answer>' in assistant_content and '</answer>' in assistant_content:
            expected = assistant_content.split('<answer>')[-1].split('</answer>')[0].strip()
        else:
            expected = assistant_content[-200:] if len(assistant_content) > 200 else assistant_content
    
    return {
        'prompt': user_content,
        'answer': str(expected)
    }

def estimate_tokens(text):
    """Rough token estimate (1 token ≈ 4 chars for English)."""
    return len(str(text)) // 4

def filter_by_length(example):
    """Filter out examples that are too long."""
    prompt_tokens = estimate_tokens(example['prompt'])
    answer_tokens = estimate_tokens(example['answer'])
    return prompt_tokens <= MAX_PROMPT_TOKENS and answer_tokens <= MAX_ANSWER_TOKENS

# === 1. Load Your HICRA Synthetic Data ===
print("📂 Loading HICRA dataset...")
my_dataset = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_train.json", 
    split="train"
)
print(f"   ✅ Loaded {len(my_dataset)} HICRA examples")

# === 2. Load Nemotron Math Data (Streaming) ===
print(f"🌊 Streaming {NEMOTRON_SAMPLE_SIZE} Nemotron math examples...")
try:
    nemotron_stream = load_dataset(
        "nvidia/Nemotron-Post-Training-Dataset-v1", 
        split="math", 
        streaming=True
    )
    
    # Take a sample and convert to list
    nemotron_list = []
    for i, example in enumerate(nemotron_stream):
        if i >= NEMOTRON_SAMPLE_SIZE:
            break
        formatted = format_nemotron(example)
        # Only keep if it's not too long
        if filter_by_length(formatted):
            nemotron_list.append(formatted)
        
        if (i + 1) % 500 == 0:
            print(f"   Processed {i + 1} examples, kept {len(nemotron_list)}...")
    
    nemotron_dataset = Dataset.from_list(nemotron_list)
    print(f"   ✅ Loaded {len(nemotron_dataset)} Nemotron examples (after length filter)")
    
except Exception as e:
    print(f"   ⚠️ Could not load Nemotron: {e}")
    print("   Continuing with HICRA data only...")
    nemotron_dataset = None

# === 3. Combine Datasets ===
print("🔀 Combining datasets...")

# Filter HICRA by length too
my_dataset_filtered = my_dataset.filter(filter_by_length)
print(f"   HICRA after filter: {len(my_dataset_filtered)} examples")

if nemotron_dataset and len(nemotron_dataset) > 0:
    from datasets import concatenate_datasets
    
    # Make sure both have the same columns
    combined_dataset = concatenate_datasets([my_dataset_filtered, nemotron_dataset])
    print(f"   ✅ Combined dataset: {len(combined_dataset)} examples")
else:
    combined_dataset = my_dataset_filtered
    print(f"   ✅ Using HICRA only: {len(combined_dataset)} examples")

# === 4. Format for GRPO Training ===
print("📝 Formatting for GRPO...")
dataset_train = combined_dataset.map(format_prompt)

# Shuffle to mix the datasets
dataset_train = dataset_train.shuffle(seed=42)

# === 5. Load Test Set (HICRA only) ===
dataset_test = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_test.json", 
    split="train"
).map(format_prompt)

print(f"\n✅ Final Training Set: {len(dataset_train)} examples")
print(f"✅ Test Set: {len(dataset_test)} examples")
print(f"\nSample prompt format:")
print(dataset_train[0]['prompt'])

📂 Loading HICRA dataset...
   ✅ Loaded 729 HICRA examples
🌊 Streaming 3000 Nemotron math examples...
   Processed 500 examples, kept 498...
   Processed 1000 examples, kept 998...
   Processed 1500 examples, kept 1498...
   Processed 2000 examples, kept 1998...
   Processed 2500 examples, kept 2498...
   Processed 3000 examples, kept 2997...
   ✅ Loaded 2997 Nemotron examples (after length filter)
🔀 Combining datasets...
   HICRA after filter: 729 examples
   ✅ Combined dataset: 3726 examples
📝 Formatting for GRPO...

✅ Final Training Set: 3726 examples
✅ Test Set: 36 examples

Sample prompt format:
[{'content': 'You are a mathematical reasoning assistant. Think through problems step by step.\nRespond in the following format:\n<think>\n...\n</think>\n<answer>\n...\n</answer>', 'role': 'system'}, {'content': 'Evaluate the integral \\(\\int_0^{2\\pi} \\sqrt{\\sin^2(t) \\cos^2(t)} \\, dt\\).', 'role': 'user'}]


In [8]:
# Cell 5: Reward Functions
import re

# Strategic reasoning phrases (from HICRA paper)
STRATEGIC_GRAMS = [
    "first i need to", "let's look at", "alternatively", "wait",
    "but i'm not sure", "let's see if", "notice that",
    "the final answer is", "let's assume", "we can conclude",
    "implies that", "to solve this", "break it down",
    "suppose that", "checking the", "recall that",
    "step 1", "step 2", "therefore", "thus"
]

def extract_xml_answer(text: str) -> str:
    """Extract answer from <answer> tags."""
    if "<answer>" not in text:
        return text.strip()
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Check if the model's answer matches the expected answer.
    Returns 2.0 for correct, 0.0 for incorrect.
    """
    responses = [completion[0]['content'] for completion in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    
    # Debug output (first item only)
    q = prompts[0][-1]['content'][:100]  # First 100 chars of question
    print(f"---\nQ: {q}...\nExpected: {answer[0]}\nExtracted: {extracted[0][:50]}...")
    
    rewards = []
    for ext, ans in zip(extracted, answer):
        # Check if answer appears in extracted text
        if str(ans).strip() in ext:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

def reasoning_reward_func(completions, **kwargs) -> list[float]:
    """
    HICRA-inspired reward for reasoning structure.
    Gives bonus for using strategic reasoning phrases.
    """
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    
    for response in responses:
        score = 0.0
        response_lower = response.lower()
        
        # Check for strategic grams
        for gram in STRATEGIC_GRAMS:
            if gram in response_lower:
                score += 0.05
        
        # Bonus for using reasoning tags
        if "<think>" in response and "</think>" in response:
            score += 0.2
        if "<answer>" in response and "</answer>" in response:
            score += 0.1
        
        # Cap the reward
        rewards.append(min(score, 0.5))
    
    return rewards

def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward for correct XML format."""
    pattern = r"<think>.*?</think>\s*<answer>.*?</answer>"
    responses = [completion[0]['content'] for completion in completions]
    return [0.5 if re.search(pattern, r, re.DOTALL) else 0.0 for r in responses]

print("✅ Reward functions defined")

✅ Reward functions defined


### Chat Template (Save for Base models)

```
# Set Llama 3 chat template (required for GRPO with conversational data)
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'system' %}<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|}
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}{% if add_generation_prompt %}<|start_header_id|>assistant<|end_header_id|>
{% endif %}"""
print("✅ Chat template set!")
```

**Optional: Use More GPU**
You could also try:

- `num_generations=6` (more diverse rollouts per step)
- Or increase `max_seq_length=1280` in cell 3 if Nemotron answers are very long

In [9]:
# Cell 7 (updated)
from trl import GRPOConfig, GRPOTrainer

# Adjusted: Give more tokens to completions
max_prompt_length = 384     # Up from 256 - plenty for most math questions
max_completion_length = 640 # 1024 - 384 = 640 tokens for reasoning

training_args = GRPOConfig(
    # Optimization
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    
    # 8 generations gives the model much better variance to learn from.
    num_generations=8, 
    
    # Keep batch size small to fit the 8 generations in memory
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8, # 1 * 8 = 8 samples per step (Standard)
    
    # LENGTH: Give it room to think!
    # 1B models often ramble. 640 is good, but 768 is safer if VRAM allows.
    max_completion_length=768,
    
    # STEPS: 1B needs repetition.
    max_steps=2500,
)

print(f"✅ Training configuration set")
print(f"   Prompt: {max_prompt_length} tokens, Completion: {max_completion_length} tokens")

✅ Training configuration set
   Prompt: 384 tokens, Completion: 640 tokens


In [10]:
# Cell 7: Initialize Trainer
print("🚀 Initializing GRPO Trainer...")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward_func,
        reasoning_reward_func,
        format_reward_func,
    ],
    args=training_args,
    train_dataset=dataset_train,
)

print("✅ Trainer initialized!")

🚀 Initializing GRPO Trainer...
✅ Trainer initialized!


Current settings `nvidia-smi` says: `6861MiB /  12282MiB`

In [11]:
# Cell 8: Run Training!
print("🏋️ Starting training...")
print("Note: First ~100 steps may show 0 reward. Be patient!")
print("="*50)

trainer_stats = trainer.train()

print("="*50)
print("✅ Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


🏋️ Starting training...
Note: First ~100 steps may show 0 reward. Be patient!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,726 | Num Epochs = 1 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!
---
Q: Solve the system of linear equations:
\[ 3x - 5y + 4z = 5 \]
\[ 7x + 2y - 3z = 2 \]
\[ 4x + 3y - 7z ...
Expected: 3) = 7 + 4 - 9 = 2\), which holds.
- For \(4x + 3y - 7z = -11\): \(4(1) + 3(2) - 7(3) = 4 + 6 - 21 = -11\), which holds.

The solution is presented as an ordered triple \((x, y, z)\).

\boxed{(1,2,3)}
Extracted: <think>
To solve the system of linear equations, w...


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / reasoning_reward_func / mean,rewards / reasoning_reward_func / std,rewards / format_reward_func / mean,rewards / format_reward_func / std
1,0.000000,0.037500,0.035355,729.250000,458.000000,768.000000,0.875000,458.000000,458.000000,458.000000,0,0,0,0,0,0.000000,0.000000,0.000000,0.037500,0.035355,0.000000,0.000000
2,0.000000,0.025000,0.037796,651.250000,232.000000,768.000000,0.750000,301.000000,232.000000,370.000000,No Log,No Log,No Log,No Log,No Log,0.000000,0.000000,0.000000,0.025000,0.037796,0.000000,0.000000
3,0.000000,0.262500,0.366206,662.375000,338.000000,768.000000,0.750000,345.500000,338.000000,353.000000,No Log,No Log,No Log,No Log,No Log,0.000260,0.000000,0.000000,0.137500,0.164208,0.125000,0.231455
4,0.000000,0.025000,0.037796,762.250000,722.000000,768.000000,0.875000,722.000000,722.000000,722.000000,No Log,No Log,No Log,No Log,No Log,0.000415,0.000000,0.000000,0.025000,0.037796,0.000000,0.000000
5,0.000000,0.081250,0.099777,703.375000,372.000000,768.000000,0.750000,509.500000,372.000000,647.000000,No Log,No Log,No Log,No Log,No Log,0.000502,0.000000,0.000000,0.081250,0.099777,0.000000,0.000000
6,0.000000,0.162500,0.298508,617.875000,297.000000,768.000000,0.500000,467.750000,297.000000,726.000000,No Log,No Log,No Log,No Log,No Log,0.000507,0.000000,0.000000,0.100000,0.122474,0.062500,0.176777
7,0.000000,0.087500,0.083452,662.875000,367.000000,768.000000,0.500000,557.750000,367.000000,728.000000,No Log,No Log,No Log,No Log,No Log,0.000389,0.000000,0.000000,0.087500,0.083452,0.000000,0.000000
8,0.000000,0.062500,0.058248,768.000000,768.000000,768.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.000398,0.000000,0.000000,0.062500,0.058248,0.000000,0.000000
9,-0.000000,2.100000,0.075593,724.875000,423.000000,768.000000,0.875000,423.000000,423.000000,423.000000,No Log,No Log,No Log,No Log,No Log,0.000466,2.000000,0.000000,0.100000,0.075593,0.000000,0.000000
10,0.000000,0.143750,0.072887,642.000000,328.000000,768.000000,0.625000,432.000000,328.000000,618.000000,No Log,No Log,No Log,No Log,No Log,0.000455,0.000000,0.000000,0.143750,0.072887,0.000000,0.000000


...

Unsloth: Input IDs of shape torch.Size([8, 1125]) with length 1125 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 1125]) with length 1125 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


---
Q: A pharmaceutical company tests the potency of a new drug across four different laboratories. Lab 1: ...
Expected: 1 and 10.0
Extracted: print("Number of laboratories with exactly one out...
---
Q: Two points are chosen independently and uniformly at random on the boundary of a square with side le...
Expected: 4/3
Extracted: Step 8: Thus, E(base)=1/2 * [x]^2 / 4= 8/2=2 thus ...
---
Q: Solve the inequality $[x](2x-1) > x^2$, where $[x]$ denotes the integer part of $x$....
Expected: he interval \([0, 2)\), the inequality does not hold, as verified in \([0, 1)\) and \([1, 2)\).

Thus, the solution set is \(x \in (-\infty, 0) \cup [2, \infty)\).

\boxed{(-\infty,0) \cup [2,\infty)}
Extracted: Therefore, in the case $0 \leq x < 1$, the subcase...
---
Q: Factor the given expressions: $a^2b^2(b-a)+b^2c^2(c-b)+a^2c^2(a-c)$....
Expected: ((c - a)\) are linear terms, and \(ab + bc + ca\) is a quadratic symmetric polynomial that is irreducible over the reals. The expression is thus complete

Unsloth: Input IDs of shape torch.Size([8, 1046]) with length 1046 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 1046]) with length 1046 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


---
Q: A risk management team is analyzing the Value at Risk (VaR) calculations for 9 different trading des...
Expected: Desk G
Extracted: Step 6: Desk G has a VaR of $8.5M, which is above ...
---
Q: Find the area of a triangle with sides 149, 163, and 222....
Expected: 195 \times 59 = 11505\), \(11505 \times 89 = 1023945\).

Thus, the area is \(12 \sqrt{1023945}\). The expression is simplified, as 1023945 has no square factors other than 1.

\boxed{12\sqrt{1023945}}
Extracted: Notice that this is an example of how we should lo...
---
Q: Solve the set of simultaneous equations
\[v^2+ w^2+ x^2+ y^2 = 6-2u,\]
\[u^2+ w^2+ x^2+ y^2 = 6-2v,\...
Expected:  variables are 1 is a valid and commonly expected solution. Given the instruction to provide a single answer and the context likely expecting this solution, the value of each variable is 1.

\boxed{1}
Extracted: Thus, by summing all the equations we get that:
\[...
---
Q: Solve the following equations: $\sin 2x-\tan(\pi /6)\cos 2x=1$....
Ex

KeyboardInterrupt: 

In [12]:
# Cell 9: Save Model
import os
# Option 1: Save locally
output_path = "llama-3.1-8b-reasoning-HICRA-v1"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"✅ Model saved to {output_path}")
# Option 2: Push to HuggingFace Hub (uncomment to use)
repo_name = "DataImaginations/Llama-3.1-8B-Reasoning-HICRA-v1"
hf_token = os.getenv('HF_TOKEN')

# Option 2: Push to HuggingFace Hub (uncomment to use)
# repo_name = "DataImaginations/Llama-1B-Reasoning-v1"
# hf_token = os.getenv('HF_TOKEN')
# 
# print(f"⏳ Pushing to {repo_name}...")
# model.push_to_hub_merged(
#     repo_name,
#     tokenizer,
#     save_method="merged_16bit",
#     token=hf_token
# )
# print("✅ Model pushed to Hub!")

✅ Model saved to llama-3.1-8b-reasoning-HICRA-v1


In [13]:
# Cell: Merge LoRA adapters and save for evaluation
from unsloth import FastLanguageModel

# Load the adapter model
model, tokenizer = FastLanguageModel.from_pretrained(
    "llama-8b-reasoning-unsloth-HICRA-v1",
    max_seq_length=1024,
    load_in_4bit=True,
)

# Merge and save in 16-bit
print("⏳ Merging adapters...")
model.save_pretrained_merged(
    "llama-3.2-8b-reasoning-merged-v1",  # New path for merged model
    tokenizer,
    save_method="merged_16bit",  # Full precision merged weights
)
print("✅ Merged model saved!")

RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

## Test the Trained Model

In [16]:
# Cell 10: Test Inference
from unsloth import FastLanguageModel

# Put model in inference mode
FastLanguageModel.for_inference(model)

# Test question
test_question = "A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": test_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=4096,
    temperature=0.7,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Question:", test_question)
print("\nResponse:")
print(response)

Question: A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment.

Response:
system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a mathematical reasoning assistant. Think through problems step by step.
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>user

A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment.assistant

<think>
Step 1: Understand the problem and the given information.
</think>
<answer>
We need to find the interest portion of the 1st payment of a loan repaid with 20 equal annual payments.
</answer>

Step 1: Recall that the interest portion of each payment is inversely proportional to the outstanding balance of the loan. Therefore, we can assum

# Bench for unsloth_HICRA

In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

# 1. Load your local model
# If you just saved your model to "llama-1b-reasoning-final", point to that folder.
print("⏳ Loading model for evaluation...")

# We wrap the model in the Harness's HFLM wrapper
# 'pretrained' can be a local path OR a Hub ID (e.g., "david-barnes/my-model")
llm = HFLM(
    pretrained="llama-8b-reasoning-merged-v1",  # Use the merged model
    batch_size=1,
    trust_remote_code=True,
    dtype="bfloat16"
)

# 2. Define the tasks you want
# These key names correspond to the harness registry.
# Note: "minerva_math" is often split by subject (algebra, etc), 
# so we usually run the main "math" group or specific subtasks.
task_list = [
    "aime24",          # AIME 2024
    "minerva_math",    # Minerva Math (covers multiple subjects)
    # "math_500",        # The 'easy' 500 questions from MATH
    "leaderboard_gpqa_main",   # leaderboard_math_hard      
]

# 3. Run the Eval
print(f"🚀 Running evaluation on: {task_list}...")
results = lm_eval.simple_evaluate(
    model=llm,
    tasks=task_list,
    num_fewshot=0,        # Reasoning models often prefer 0-shot (Instruction)
    limit=None,           # Set to e.g., 50 to test quickly before full run!
    log_samples=True,    # Set True if you want to see exactly what it got wrong
)

# 4. Print a Pretty Table
from lm_eval.utils import make_table
print(make_table(results))

# 5. Save detailed results to JSON (Crucial for your blog!)
import json
with open("llama_8b_unsloth_HICRA_v1_benchmark_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Soft VRAM clear

In [ ]:
import torch
import gc

# 1. Delete the Python variables holding the model
# (Wrap in try/except so it doesn't crash if they are already gone)
try:
    del model
    del tokenizer
    del trainer
except NameError:
    print("Variables already deleted or not defined.")

# 2. Python Garbage Collection (Clears CPU RAM)
gc.collect()

# 3. PyTorch Cache Clearing (The most important step for VRAM)
torch.cuda.empty_cache()

# Verify: Print current memory usage
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")